In [78]:
import os


#DEfinição das camadas

folders = [

           'data/bronze', #Arquivo original sem alterações
           'data/silver',  #Arquivo limpo e padronizados
           'data/gold',    #Arquivo com tabelas agregadas para o Looker Studio
           'logs'          #Registro de execução
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"Diretório {folder} pronto.")

Diretório data/bronze pronto.
Diretório data/silver pronto.
Diretório data/gold pronto.
Diretório logs pronto.


In [79]:
import logging

import pandas as pd
import re
from sqlalchemy import create_engine

#Criando Pasta

for folder in ['data/bronze', 'data/silver', 'data/gold', 'logs']:
    os.makedirs(folder,exist_ok=True)

#Congifuração para corrigir o Logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("logs/pipeline.log"),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(__name__)
logger.info("Infraestrutura iniciada com sucesso.")

In [80]:
def extract_data(file_path):

  #Lê o arquivo original e registra o início e fim.

  try:
      logger.info("Iniciando a extração dos dados ")


      #Carregando o arquivo Excel

      df_raw = pd.read_excel(file_path)

      #Salvando uma códia na pasta Bronze

      bronze_path = "data/bronze/dados_originais.csv"
      df_raw.to_csv(bronze_path, index=False)

      logger.info(f"Extração concluida comsucesso do Arquivo salvo em: {bronze_path}")
      return df_raw
  except Exception as e:
      logger.error(f"Erro durante a extração: {e}")
      return None



In [81]:
def extract_bronze():
    try:
        logger.info("Iniciando a extração dos arquivos CSV (Camada Bronze)...")

        # Lendo os três arquivos

        vendas = pd.read_csv('fato_vendas.csv')
        produtos = pd.read_csv('dim_produto.csv')
        filiais = pd.read_csv('dim_filial.csv')

        # Salvando cópias na pasta Bronze

        vendas.to_csv("data/bronze/fato_vendas_raw.csv", index=False)
        produtos.to_csv("data/bronze/dim_produto_raw.csv", index=False)
        filiais.to_csv("data/bronze/dim_filial_raw.csv", index=False)

        logger.info("Sucesso! Os 3 arquivos foram copiados para data/bronze")
        return vendas, produtos, filiais
    except Exception as e:
        logger.error(f"Erro durante a extração: {e}")
        return None, None, None

# Chamando a função

df_vendas, df_produtos, df_filiais = extract_bronze()

In [82]:

def transform_silver(vendas, produtos, filiais):
    if vendas is None: return None

    try:
        logger.info("Iniciando união das tabelas")


      # Unindo as tabelas

        df = vendas.merge(produtos, on='produto_id', how='left')
        df = df.merge(filiais, on='filial_id' , how='left')

      # Padronizando nomes

        df.columns = [re.sub(r'\W+', '_', col.strip().lower()) for col in df.columns]

      # Salvando na Silver

        df.to_csv("data/silver/vendas_consolidadas.csv",index=False)

        logger.info("Camada Silver concluída com sucesso!")
        return df
    except Exception as e:
        logger.error(f"Erro durante a transformação: {e}")
        return None

# Executa a limpeza e união

df_silver = transform_silver(df_vendas, df_produtos, df_filiais)

In [84]:
import pandas as pd

def transform_gold(df_silver):
    try:
        logger.info("Iniciando a Camada Gold")

        # 1. Garantir que os dados são números (resolve o erro de str + int)

        df_prep = df_silver.copy()
        df_prep['receita'] = pd.to_numeric(df_prep['receita'], errors='coerce').fillna(0)

        # 2. Criar colunas separadas

        df_prep['vendas_clamed'] = df_prep.apply(lambda x: x['receita'] if str(x['empresa']).strip() == 'Clamed' else 0, axis=1)
        df_prep['vendas_concorrencia'] = df_prep.apply(lambda x: x['receita'] if str(x['empresa']).strip() == 'Concorrente' else 0, axis=1)

        # 3. Agrupar os dados

        df_gold = df_prep.groupby(['data', 'brick', 'regiao', 'categoria']).agg({
            'vendas_clamed': 'sum',
            'vendas_concorrencia': 'sum'
        }).reset_index()

        # 4. Cálculo do Market Share

        df_gold['total_mercado'] = df_gold['vendas_clamed'] + df_gold['vendas_concorrencia']
        df_gold['market_share_clamed'] = (df_gold['vendas_clamed'] / df_gold['total_mercado'].replace(0, 1)) * 100

        # 5. Salvando o arquivo

        gold_path = "data/gold/dashboard_clamed.csv"
        df_gold.to_csv(gold_path, index=False)

        print(f"🎉 SUCESSO! Arquivo gerado em: {gold_path}")
        return df_gold

    except Exception as e:
        print(f"❌ ERRO: {e}")
        return None

